# Intent classifier, Option B: fine-tuned XLM-RoBERTa

Same 690 sentences and the same 5-fold `StratifiedGroupKFold` split by template as Option A (MiniLM + LogisticRegression), so the results can be compared directly.

**Input:** `intent_dataset.csv` (columns `text`, `label`, `group`) uploaded as a Kaggle dataset. **Accelerator:** GPU T4.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: 2 GPUs halve the training steps
import glob, time
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

MODEL = "xlm-roberta-base"
LABELS = ["data", "forecast", "orders", "chat", "personal_data", "write", "off_topic"]
L2I = {l: i for i, l in enumerate(LABELS)}

path = glob.glob("/kaggle/input/**/intent_dataset.csv", recursive=True)[0]
df = pd.read_csv(path)
df["y"] = df["label"].map(L2I)
print(len(df), "frases"); print(df["label"].value_counts())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SEM GPU")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(frame):
    ds = Dataset.from_dict({"text": frame["text"].tolist(), "label": frame["y"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *a, **kw):
        super().__init__(*a, **kw); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels,
                                                 weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def train_model(train_df, out_dir):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=20, learning_rate=3e-5,
        per_device_train_batch_size=16, warmup_steps=60, weight_decay=0.01,
        fp16=True, save_strategy="no", logging_steps=50, report_to="none", seed=42)
    # class weights: small classes (chat, write, off_topic) count more in the loss
    counts = np.bincount(train_df["y"], minlength=len(LABELS))
    weights = torch.tensor(len(train_df) / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float)
    trainer = WeightedTrainer(weights, model=model, args=args, train_dataset=to_ds(train_df),
                              data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    return trainer

def predict_proba(trainer, frame):
    out = trainer.predict(to_ds(frame))
    return torch.softmax(torch.tensor(out.predictions), dim=-1).numpy()

## 5-fold cross-validation (split by template)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
proba = np.zeros((len(df), len(LABELS)))
t0 = time.time()
for k, (tr, te) in enumerate(cv.split(df["text"], df["y"], df["group"]), 1):
    print(f"--- fold {k}: treino {len(tr)}, teste {len(te)}")
    trainer = train_model(df.iloc[tr], f"/tmp/fold{k}")
    proba[te] = predict_proba(trainer, df.iloc[te])
    del trainer; torch.cuda.empty_cache()
print(f"Tempo total: {(time.time()-t0)/60:.1f} min")

In [ ]:
y_true = df["y"].values
y_pred = proba.argmax(1); conf = proba.max(1)

lines = ["Classificador de intenção: XLM-RoBERTa base (fine-tune)",
         f"Avaliação: StratifiedGroupKFold (5 folds, split por template), {len(df)} frases", "",
         classification_report(y_true, y_pred, target_names=LABELS, digits=3),
         "Matriz de confusão (linhas = real, colunas = previsto):",
         str(pd.DataFrame(confusion_matrix(y_true, y_pred), index=LABELS, columns=[l[:8] for l in LABELS])),
         "", "Limiar | decididas pelo classificador | accuracy nessas"]
for th in (0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9):
    s = conf >= th
    lines.append(f"  {th:.2f} | {s.mean():6.1%} | {(y_true[s] == y_pred[s]).mean():6.1%}")
report = "\n".join(lines)
print(report)
open("/kaggle/working/intent_report_xlmr.txt", "w", encoding="utf-8").write(report)

## Final model on all 690 sentences + CPU latency

In [ ]:
final = train_model(df, "/tmp/final")
final.save_model("/kaggle/working/intent-xlmr"); tok.save_pretrained("/kaggle/working/intent-xlmr")

cpu_model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/intent-xlmr").eval()
q = "Quanto faturou Braga em agosto?"
with torch.no_grad():
    for _ in range(3): cpu_model(**tok(q, return_tensors="pt"))
    t0 = time.time()
    for _ in range(20): cpu_model(**tok(q, return_tensors="pt"))
print(f"Latência em CPU: {(time.time()-t0)/20*1000:.0f} ms por pergunta")

In [ ]:
!cd /kaggle/working && zip -qr intent-xlmr.zip intent-xlmr && ls -lh intent-xlmr.zip intent_report_xlmr.txt